# CAN Bus Anomaly Detection — Autoencoder

Dataset: can-train-and-test (Lampe & Meng 2024) — set_01

Trains a dense autoencoder over per-frame features (ID behavioral stats + payload) on **normal traffic only**.
Outputs `best_model.pth` + `top_can_ids.json` to `/kaggle/working` for download.

In [ ]:
import sys
import json
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ── Paths ──
# Change to your uploaded Kaggle dataset path containing ae_data.npz
DATA_DIR = Path("/kaggle/input/can-anomaly-detection-data")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ──
TOP_K = 30
INPUT_DIM = 13
EPOCHS = 50
BATCH_SIZE = 256
LR = 0.001
WEIGHT_DECAY = 1e-5
PATIENCE = 10
VAL_RATIO = 0.2
THRESHOLD_PERCENTILE = 99
SEED = 42

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed()

---
## Model: Dense autoencoder (no CAN ID embedding)

```
Features (13: 4 ID-behavioral stats + 8 bytes + DLC)
    └──→ 13 → 8 → 4 (bottleneck) → 8 → 13 → Tanh
```

- No dropout (fit normal data closely)
- Output: Tanh (inputs are in [-0.004, 1])
- Loss: MSE between input (13-dim) and reconstruction (13-dim)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM):
        super().__init__()
        self.input_dim = input_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
        )

        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


---
## Dataset

In [ ]:
class AEMDataset(Dataset):
    def __init__(self, features):
        self.features = torch.tensor(features, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]


# The .npz already contains the 80/20 chronological split (normal data only)
d = np.load(DATA_DIR / "ae_data.npz", allow_pickle=True)
top_ids = d["top_ids"].tolist()
print(f"top_ids ({len(top_ids)}): {top_ids[:5]} ...")
print(f"train feats: {d['train_feats'].shape}  val feats: {d['val_feats'].shape}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

NUM_WORKERS = 0 if sys.platform == "win32" else 2
loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(
    AEMDataset(d["train_feats"]),
    shuffle=True, drop_last=True, **loader_kw,
)
val_loader = DataLoader(AEMDataset(d["val_feats"]), **loader_kw)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

model = Autoencoder(input_dim=INPUT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Training

In [ ]:
def run_epoch(model, loader, device, optimizer=None, desc=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    losses = []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    pbar = tqdm(loader, desc=desc or ("train" if is_train else "val"), leave=False)
    with ctx:
        for feats in pbar:
            feats = feats.to(device)
            recon = model(feats)
            loss = F.mse_loss(recon, feats)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.6f}")
    return np.mean(losses)


@torch.no_grad()
def compute_threshold(model, loader, device, percentile=THRESHOLD_PERCENTILE):
    model.eval()
    all_errs = []
    for feats in tqdm(loader, desc="Threshold", leave=False):
        feats = feats.to(device)
        recon = model(feats)
        mse = (recon - feats).pow(2).mean(dim=1)
        all_errs.append(mse.cpu().numpy())
    all_errs = np.concatenate(all_errs)
    return float(np.percentile(all_errs, percentile))

In [ ]:
print(f"\n{'=' * 60}")
print(f"Training for {EPOCHS} epochs (patience={PATIENCE})")
print(f"{'=' * 60}")

history = {"epoch": [], "train_loss": [], "val_loss": []}
best_val_loss = float("inf")
stale = 0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, device, optimizer, desc=f"E{epoch} train")
    val_loss = run_epoch(model, val_loader, device, desc=f"E{epoch} val")

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        stale = 0
        marker = " *"
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model_weights.pth")
    else:
        stale += 1

    print(f"Epoch {epoch:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f}{marker}")

    if stale >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print("\nComputing threshold on validation normal data ...")
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model_weights.pth", map_location=device))
threshold = compute_threshold(model, val_loader, device)
print(f"Threshold ({THRESHOLD_PERCENTILE}th pct): {threshold:.6f}")

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"\nDone! Best val loss: {best_val_loss:.6f} at epoch {best_epoch} | Threshold: {threshold:.6f}")

In [ ]:
# Save final checkpoint + top CAN IDs for local evaluation
torch.save({
    "epoch": best_epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": best_val_loss,
    "threshold": threshold,
}, OUTPUT_DIR / "best_model.pth")

with open(OUTPUT_DIR / "top_can_ids.json", "w") as f:
    json.dump(top_ids, f)

print("Saved:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  /kaggle/working/{f.name} ({f.stat().st_size / 1e6:.2f} MB)")

---
## Loss Curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["epoch"], history["train_loss"], lw=2, label="Train")
ax.plot(history["epoch"], history["val_loss"], lw=2, ls="--", label="Validation")
ax.set(xlabel="Epoch", ylabel="MSE Loss", title="Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()